Install packages and spaCy model

In [ ]:
!pip -q install pandas pyarrow spacy datasets
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 94.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


Mount Drive and load data

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import pandas as pd

BASE_PATH = Path("/content/drive/MyDrive/AI-Resume-Screener")

train_df = pd.read_parquet(BASE_PATH / "data/processed/train.parquet")
val_df = pd.read_parquet(BASE_PATH / "data/processed/validation.parquet")
test_df = pd.read_parquet(BASE_PATH / "data/processed/test.parquet")

Mounted at /content/drive


Preprocessing function

In [ ]:
import re
import spacy

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

def basic_clean(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def preprocess_batch(texts):
    cleaned_texts = [basic_clean(text) for text in texts]
    results = []

    for doc in nlp.pipe(cleaned_texts, batch_size=64):
        tokens = [
            token.lemma_
            for token in doc
            if not token.is_stop
            and not token.is_punct
            and not token.is_space
        ]
        results.append(" ".join(tokens))

    return results

Preprocess a manageable baseline sample

In [ ]:
BASELINE_TRAIN_SIZE = min(15000, len(train_df))

baseline_train_df = train_df.sample(
    n=BASELINE_TRAIN_SIZE,
    random_state=42,
).reset_index(drop=True)

baseline_test_df = test_df.sample(
    n=min(2000, len(test_df)),
    random_state=42,
).reset_index(drop=True)

baseline_train_df["clean_resume"] = preprocess_batch(baseline_train_df["resume"])
baseline_train_df["clean_jd"] = preprocess_batch(baseline_train_df["jd"])

baseline_test_df["clean_resume"] = preprocess_batch(baseline_test_df["resume"])
baseline_test_df["clean_jd"] = preprocess_batch(baseline_test_df["jd"])

Skill extraction using PhraseMatcher

In [ ]:
from spacy.matcher import PhraseMatcher

SKILLS = [
    "python", "java", "javascript", "typescript", "c++", "c#",
    "sql", "mysql", "postgresql", "mongodb", "redis",
    "machine learning", "deep learning", "natural language processing",
    "nlp", "computer vision", "data analysis", "data science",
    "tensorflow", "pytorch", "scikit-learn", "pandas", "numpy",
    "docker", "kubernetes", "git", "linux", "aws", "azure", "gcp",
    "react", "angular", "node.js", "django", "flask", "fastapi",
    "power bi", "tableau", "excel", "project management",
    "communication", "leadership", "problem solving",
]

skill_nlp = spacy.blank("en")
matcher = PhraseMatcher(skill_nlp.vocab, attr="LOWER")
matcher.add("SKILL", [skill_nlp.make_doc(skill) for skill in SKILLS])

def extract_skills(text):
    doc = skill_nlp(str(text))
    matches = matcher(doc)
    return sorted({doc[start:end].text.lower() for _, start, end in matches})

def compare_skills(resume, jd):
    resume_skills = set(extract_skills(resume))
    jd_skills = set(extract_skills(jd))

    return {
        "resume_skills": sorted(resume_skills),
        "jd_skills": sorted(jd_skills),
        "matched_skills": sorted(resume_skills & jd_skills),
        "missing_skills": sorted(jd_skills - resume_skills),
    }

display(compare_skills(
    baseline_test_df.loc[0, "resume"],
    baseline_test_df.loc[0, "jd"],
))

{'resume_skills': ['c #', 'c#', 'communication', 'java', 'javascript', 'sql'],
 'jd_skills': ['angular', 'azure', 'c#', 'communication', 'typescript'],
 'matched_skills': ['c#', 'communication'],
 'missing_skills': ['angular', 'azure', 'typescript']}

Save preprocessed baseline data

In [ ]:
baseline_train_df.to_parquet(
    BASE_PATH / "data/processed/baseline_train.parquet",
    index=False,
)
baseline_test_df.to_parquet(
    BASE_PATH / "data/processed/baseline_test.parquet",
    index=False,
)

Load a skill-labelled dataset for NER evaluation

In [ ]:
from datasets import load_dataset

skill_dataset = load_dataset("batuhanmtl/job_resume_fit", split="train")
skill_df = skill_dataset.to_pandas()

print(skill_df.columns.tolist())
display(skill_df.head(1))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/1.36k [00:00<?, ?B/s]

job_resume_fit.csv:   0%|          | 0.00/30.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2385 [00:00<?, ? examples/s]

['ID', 'resume_text', 'job_text', 'category', 'job_required_skills', 'resume_skill_list', 'ai_matched_skills', 'ai_match_score', 'skill_string_match_score', 'fuzzy_match_score']


,ID,resume_text,job_text,category,job_required_skills,resume_skill_list,ai_matched_skills,ai_match_score,skill_string_match_score,fuzzy_match_score
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,Human Resources Business Partner\n------------...,HR,"['human resources management', 'talent acquisi...","['customer service', 'team management', 'marke...","['human resources management', 'employee relat...",49.4,12.5,63.24
